<a href="https://colab.research.google.com/github/Ankitha1089/Marketing-Campaign-Analysis/blob/main/MarketingSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sqlite3


In [ ]:
df = pd.read_csv("/content/marketing_data_segmentation.csv")


In [ ]:
conn = sqlite3.connect("marketing.db")
cursor = conn.cursor()


In [ ]:
df.to_sql("customers", conn, if_exists="replace", index=False)


54984

In [ ]:
cursor.execute("PRAGMA table_info(customers);")
cursor.fetchall()


[(0, 'ID', 'INTEGER', 0, None, 0),
 (1, 'Year_Birth', 'INTEGER', 0, None, 0),
 (2, 'Education', 'TEXT', 0, None, 0),
 (3, 'Marital_Status', 'TEXT', 0, None, 0),
 (4, 'Income', 'REAL', 0, None, 0),
 (5, 'Kidhome', 'INTEGER', 0, None, 0),
 (6, 'Teenhome', 'INTEGER', 0, None, 0),
 (7, 'Dt_Customer', 'TEXT', 0, None, 0),
 (8, 'Recency', 'INTEGER', 0, None, 0),
 (9, 'MntWines', 'INTEGER', 0, None, 0),
 (10, 'MntFruits', 'INTEGER', 0, None, 0),
 (11, 'MntMeatProducts', 'INTEGER', 0, None, 0),
 (12, 'MntFishProducts', 'INTEGER', 0, None, 0),
 (13, 'MntSweetProducts', 'INTEGER', 0, None, 0),
 (14, 'MntGoldProds', 'INTEGER', 0, None, 0),
 (15, 'NumDealsPurchases', 'INTEGER', 0, None, 0),
 (16, 'NumWebPurchases', 'INTEGER', 0, None, 0),
 (17, 'NumCatalogPurchases', 'INTEGER', 0, None, 0),
 (18, 'NumStorePurchases', 'INTEGER', 0, None, 0),
 (19, 'NumWebVisitsMonth', 'INTEGER', 0, None, 0),
 (20, 'AcceptedCmp3', 'INTEGER', 0, None, 0),
 (21, 'AcceptedCmp4', 'INTEGER', 0, None, 0),
 (22, 'AcceptedC

In [ ]:
#Total customers

pd.read_sql("SELECT COUNT(*) FROM customers;", conn)



,COUNT(*)
0,54984


In [ ]:
#Check missing income

pd.read_sql("""
SELECT COUNT(*) AS missing_income
FROM customers
WHERE Income IS NULL;
""", conn)


,missing_income
0,0


In [ ]:
#SQL VIEW CREATION

cursor.execute("""
CREATE VIEW customer_segments AS
SELECT
    *,

    -- High income customers
    CASE
        WHEN Income > 75000 THEN 1 ELSE 0
    END AS High_Income,

    -- Young customers
    CASE
        WHEN Age < 30 THEN 1 ELSE 0
    END AS Young_Customer,

    -- Family customers
    CASE
        WHEN Children > 0 THEN 1 ELSE 0
    END AS Family_Customer,

    -- Campaign responders
    CASE
        WHEN Response = 1 THEN 1 ELSE 0
    END AS Campaign_Responder,

    -- High web engagement
    CASE
        WHEN NumWebVisitsMonth > 5 THEN 1 ELSE 0
    END AS High_Web_Engagement,

    -- High spenders (data-driven)
    CASE
        WHEN Total_Spend >
             (SELECT AVG(Total_Spend) FROM customers)
        THEN 1 ELSE 0
    END AS High_Spender

FROM customers;
""")

conn.commit()


In [ ]:
#Segment counts

pd.read_sql("""
SELECT
    SUM(High_Income) AS High_Income,
    SUM(Young_Customer) AS Young_Customers,
    SUM(Family_Customer) AS Family_Customers,
    SUM(High_Spender) AS High_Spenders
FROM customer_segments;
""", conn)


,High_Income,Young_Customers,Family_Customers,High_Spenders
0,19374,0,37614,5490


In [ ]:
#Campaign response by segment

pd.read_sql("""
SELECT
    High_Spender,
    COUNT(*) AS Customers,
    ROUND(AVG(Response), 2) AS Response_Rate
FROM customer_segments
GROUP BY High_Spender;
""", conn)


,High_Spender,Customers,Response_Rate
0,0,49494,0.13
1,1,5490,0.26


In [ ]:
#Channel behavior by high spenders

pd.read_sql("""
SELECT
    High_Spender,
    AVG(NumDealsPurchases) AS Avg_Deals,
    AVG(NumWebPurchases) AS Avg_Web,
    AVG(NumStorePurchases) AS Avg_Store,
    AVG(NumCatalogPurchases) AS Avg_Catalog
FROM customer_segments
GROUP BY High_Spender;
""", conn)


,High_Spender,Avg_Deals,Avg_Web,Avg_Store,Avg_Catalog
0,0,2.193276,4.107387,4.523195,1.983149
1,1,1.984517,5.459927,6.286885,3.200729


In [ ]:
#Which segment responds most to campaigns

pd.read_sql("""
SELECT
    High_Income,
    ROUND(AVG(Response), 2) AS Response_Rate
FROM customer_segments
GROUP BY High_Income;
""", conn)


,High_Income,Response_Rate
0,0,0.10
1,1,0.23


In [ ]:
#Spending by marital status

pd.read_sql("""
SELECT
    Marital_Status,
    ROUND(AVG(Total_Spend), 2) AS Avg_Spend
FROM customer_segments
GROUP BY Marital_Status
ORDER BY Avg_Spend DESC;
""", conn)


,Marital_Status,Avg_Spend
0,Divorced,749.50
1,Widow,739.51
2,Alone,699.33
3,Single,682.67
4,Yolo,661.67
5,Absurd,656.51
6,Together,640.74
7,Married,501.90


In [ ]:
#Country-wise high-value customers

pd.read_sql("""
SELECT
    Country,
    COUNT(*) AS High_Value_Customers
FROM customer_segments
WHERE High_Spender = 1
GROUP BY Country
ORDER BY High_Value_Customers DESC;
""", conn)

,Country,High_Value_Customers
0,Spain,1343
1,Saudi Arabia,868
2,Canada,861
3,Australia,825
4,India,565
5,Germany,481
6,Usa,470
7,Mexico,77


In [ ]:
df.to_sql("customers", conn, if_exists="replace", index=False)
conn.close()


In [ ]:
from google.colab import files
files.download("marketing.db")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.isna().sum()

,0
ID,0
Year_Birth,0
Education,0
Marital_Status,0
Income,0
Kidhome,0
Teenhome,0
Dt_Customer,0
Recency,0
MntWines,0
